In [ ]:
# To enable horizontal scrolling
from IPython.display import display, HTML
display(HTML("<style>pre { white-space: pre !important; }</style>"))

In [ ]:
%env SPARK_HOME=/opt/spark

# Download of NYC taxi trips and taxi zone file

In [ ]:
base_directory = "./data"

In [ ]:
!pip install wget findspark leafmap geojson

In [ ]:
import os
import wget
import zipfile

base_directory = os.path.abspath(base_directory)
os.environ["BASEDIRECTORY"] = base_directory

# Download yellow trip data (January 2022)
data_directory = base_directory + "/taxidata"
data_file = "yellow_tripdata_2022-01.parquet"
data_path = data_directory + "/" + data_file
if not os.path.exists(data_path):
    os.makedirs(data_directory, exist_ok=True)
if not os.path.exists(data_path):
    wget.download("https://d37ci6vzurychx.cloudfront.net/trip-data/" + data_file, out=data_directory)

# Download zone shapefile and lookup
zone_directory = base_directory + "/taxizonesdata"
if not os.path.isdir(zone_directory):
    os.makedirs(zone_directory, exist_ok=True)

zone_zipfile = "taxi_zones.zip"
zone_zipfile_path = zone_directory + "/" + zone_zipfile
if not os.path.exists(zone_zipfile_path):
    wget.download("https://d37ci6vzurychx.cloudfront.net/misc/" + zone_zipfile, out=zone_directory)
    with zipfile.ZipFile(zone_zipfile_path, "r") as zip_ref:
        zip_ref.extractall(zone_directory)
        zip_ref.close()

zone_lookup_file = "taxi_zone_lookup.csv"
if not os.path.exists(zone_directory + "/" + zone_lookup_file):
    wget.download("https://d37ci6vzurychx.cloudfront.net/misc/" + zone_lookup_file, out=zone_directory)

# Initialisation of Spark context

**What is Spark?**  
Apache Spark is a distributed data processing framework. It processes data in parallel across a cluster (or locally on your machine).  
- `SparkSession` — the entry point. Creates DataFrames, runs SQL, reads files.
- `SparkContext (sc)` — low-level context for RDD operations.
- **DataFrame** — distributed table, like a SQL table or pandas DataFrame but potentially millions of rows.

In [ ]:
import findspark
import os
findspark.init(os.environ['SPARK_HOME'])
print(os.environ['SPARK_HOME'])

In [ ]:
import platform
import pyspark
from pyspark import SparkContext
from pyspark.sql import SparkSession

# Create a SparkSession — the main entry point
spark = SparkSession.builder \
    .appName("Python Spark Map Visualization of NYC taxi trips") \
    .getOrCreate()

sc = spark.sparkContext
print(f"Spark version: {spark.version}")

In [ ]:
# Load the parquet file as a Spark DataFrame
# Parquet = columnar binary format — much faster than CSV for analytics
trips = spark.read.parquet(data_path)
print(f"Loaded {trips.count():,} trips")

In [ ]:
# Inspect the schema: column names and their data types
# dtypes returns a list of (column_name, type_string) tuples
trips.dtypes

In [ ]:
# Show first 20 rows
trips.show()

# Grouping using groupBy

`groupBy(column).count()` — groups rows by a column's unique values and counts each group.  
Equivalent SQL: `SELECT VendorID, COUNT(*) FROM trips GROUP BY VendorID`

In [ ]:
# Example: count trips by VendorID (1 = Creative Mobile Technologies, 2 = VeriFone)
trips.groupBy("VendorID").count().show()

# ✅ Exercise 1 — Count trips grouped by number of passengers

**Task:** Group trips by `passenger_count` and count how many trips have each number of passengers.

**Expected unexpected values:**
- `0` passengers — "ghost" trips or driver errors; or vehicle repositioning without a passenger
- Very high counts (7, 8, 9) — possibly data entry errors; NYC yellow taxis max capacity is 4
- `NULL` — missing data; some meters don't record passenger count

**Reference:** https://www1.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_yellow.pdf

In [ ]:
# ✅ Exercise 1 Solution
# Group by passenger_count and sort by passenger_count ascending to see the full range
(
    trips
    .groupBy("passenger_count")
    .count()
    .orderBy("passenger_count")
    .show(20)
)
# Unexpected values to discuss:
# - 0 passengers: trips without passengers (driver repositioning, meter error)
# - NULL: missing data — some vendors don't fill this field
# - >6 passengers: data quality issue — NYC taxis max capacity is 6

# ✅ Exercise 2 — Minimum distance per passenger group

**Task:** Find the **minimum trip distance** for each passenger count group.

**Expected unexpected values:**
- Minimum of `0.0` miles in most groups — zero-distance trips exist (cancelled or meter errors)
- Very small negative values are possible in some datasets (sensor noise)

In [ ]:
# ✅ Exercise 2 Solution
# Group by passenger_count, then aggregate with min() on trip_distance
from pyspark.sql import functions as F

(
    trips
    .groupBy("passenger_count")
    .agg(F.min("trip_distance").alias("min_distance"))
    .orderBy("passenger_count")
    .show(20)
)
# Almost all groups will show 0.0 miles minimum
# These are zero-distance trips — likely cancelled, refunded, or data errors

# ✅ Exercise 3 — Remove 0.0-mile trips, recompute minimum

**Task:** Filter out `trip_distance == 0.0`, then recompute the minimum distance per group.

**Expected result:** All minimum distances > 0. They'll likely be very small (0.01–0.1 miles) — very short trips within a block.

In [ ]:
# ✅ Exercise 3 Solution
# Chain: filter out 0.0 miles → groupBy → min
(
    trips
    .filter(trips.trip_distance > 0.0)        # remove zero-distance trips
    .groupBy("passenger_count")
    .agg(F.min("trip_distance").alias("min_nonzero_distance"))
    .orderBy("passenger_count")
    .show(20)
)
# Now all minimum values are > 0.0
# Very short trips (0.01 miles) are still valid — crosstown blocks in Manhattan

# SQL Queries

Spark SQL lets you run standard SQL queries on DataFrames.  
Step 1: Register the DataFrame as a **temporary view** (like a SQL table name).  
Step 2: Run `sqlContext.sql("SELECT ...")` to query it.

In [ ]:
from pyspark.sql.types import *
sqlContext = SparkSession.builder.getOrCreate()

In [ ]:
# Register trips DataFrame as SQL table named "trips"
trips.createOrReplaceTempView("trips")

In [ ]:
# Example: select fare_amount for trips >= 5 miles
query = "SELECT fare_amount FROM trips WHERE trip_distance >= 5"
sqlContext.sql(query).show()

# ✅ Exercise 4 — Rewrite SQL as functional (DataFrame API)

**Task:** Rewrite `SELECT fare_amount FROM trips WHERE trip_distance >= 5` using the Spark DataFrame API (no SQL string).

**Key methods:**
- `.filter()` or `.where()` — equivalent to SQL `WHERE`
- `.select()` — equivalent to SQL `SELECT`

In [ ]:
# ✅ Exercise 4 Solution
# Method 1: using column reference
trips.filter(trips.trip_distance >= 5).select("fare_amount").show()

# Method 2: using string column name
trips.where("trip_distance >= 5").select("fare_amount").show()

# Method 3: using col() function
from pyspark.sql.functions import col
trips.filter(col("trip_distance") >= 5).select(col("fare_amount")).show()

In [ ]:
# Compute summary statistics for all columns (count, mean, stddev, min, max)
trips.describe().show()

# ✅ Exercise 5 — SQL: trip distances for tips > $5

**Task:** Write a SQL query to get the `trip_distance` for all trips where `tip_amount > 5`.

In [ ]:
# ✅ Exercise 5 Solution
query = "SELECT trip_distance FROM trips WHERE tip_amount > 5"
sqlContext.sql(query).show()

# Also show summary stats to understand the distribution
sqlContext.sql("""
    SELECT
        COUNT(*) AS num_trips,
        MIN(trip_distance)  AS min_dist,
        AVG(trip_distance)  AS avg_dist,
        MAX(trip_distance)  AS max_dist
    FROM trips
    WHERE tip_amount > 5
""").show()
# Expected: trips with high tips tend to be longer-distance trips

# ✅ Exercise 6 — SQL: total amount for distances > 30 miles

**Task:** Formulate a SQL query to get the `total_amount` for trips with `trip_distance > 30`.

In [ ]:
# ✅ Exercise 6 Solution
query = "SELECT total_amount FROM trips WHERE trip_distance > 30"
sqlContext.sql(query).show()

# Enhanced version: show count and avg total_amount for context
sqlContext.sql("""
    SELECT
        COUNT(*)           AS num_trips,
        AVG(total_amount)  AS avg_total_amount,
        MAX(total_amount)  AS max_total_amount,
        MIN(trip_distance) AS min_dist
    FROM trips
    WHERE trip_distance > 30
""").show()
# These are airport runs (JFK/LaGuardia) or long suburban trips

# ✅ Exercise 7 — Box-and-Whisker Plots of Numerical Columns

**Task:** For each numerical column, compute the 5-number summary and plot a box-and-whisker plot.

**5-number summary** (quantiles at [0.0, 0.25, 0.5, 0.75, 1.0]):
```
whislo = Q0   (minimum / lower whisker)
q1     = Q25  (1st quartile)
med    = Q50  (median)
q3     = Q75  (3rd quartile)
whishi = Q100 (maximum / upper whisker)
```

**Key method:** `trips.stat.approxQuantile(colName, quantiles, relativeError)`  
- `relativeError=0.01` → 1% relative accuracy (fast approximation)

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

In [ ]:
# ✅ Exercise 7 Solution
for column in trips.dtypes:
    name = column[0]
    colType = column[1]

    # Only process numerical columns (skip strings and timestamps)
    if colType not in ('string', 'timestamp', 'timestamp_ntz'):

        # approxQuantile returns [q0, q25, q50, q75, q100]
        # relativeError=0.01 means ~1% accuracy — much faster than exact
        columnQuantiles = trips.stat.approxQuantile(
            name,
            [0.0, 0.25, 0.5, 0.75, 1.0],
            0.01   # relative error
        )
        print("{} quantiles: {}".format(name, columnQuantiles))

        # Build the bxpstats dict required by matplotlib's bxp()
        stats = [{
            "whislo": columnQuantiles[0],   # Q0  (minimum)
            "q1":     columnQuantiles[1],   # Q25
            "med":    columnQuantiles[2],   # Q50 (median)
            "q3":     columnQuantiles[3],   # Q75
            "whishi": columnQuantiles[4],   # Q100 (maximum)
        }]

        fig, axes = plt.subplots(nrows=1, ncols=1, figsize=(5, 5), sharey=True)
        axes.bxp(bxpstats=stats, showfliers=False)
        axes.grid(True)
        axes.set_title(name)
        plt.tight_layout()
        plt.show()

# What these plots tell you:
# - Narrow box + long whisker upward → most trips are short, but rare very long ones
# - Median near bottom of box → right-skewed distribution
# - fare_amount, tip_amount, total_amount all show similar right-skewed patterns

# ✅ Exercise 8 — Trips per Weekday

**Task:** Count the number of trips per day of the week, then visualize as a bar chart.

**Key concepts:**
- `@udf` — **User Defined Function**: a Python function that Spark can apply to each row
- `weekday()`: returns 0=Monday … 6=Sunday (Python's `datetime.weekday()`)
- `weekdayStr()`: returns the actual day name ("Monday", etc.)

In [ ]:
def barchart(dataRows, titleSuffix):
    positions = list(reversed(range(len(dataRows))))
    names = [str(item[titleSuffix]) + " (" + str(item['count']) + ")" for item in dataRows]
    values = [item['count'] for item in dataRows]
    plt.figure(figsize=(10, 5))
    plt.grid()
    plt.barh(positions, values, align="center")
    plt.yticks(positions, names)
    plt.xlabel("Number of trips")
    plt.title("Distribution of trips per " + titleSuffix)
    plt.tight_layout()
    plt.show()

In [ ]:
from pyspark.sql.functions import udf
from pyspark.sql.functions import col
from pyspark.sql.types import IntegerType, StringType
import calendar

# UDF returning day name string ("Monday", "Tuesday", ...)
@udf(returnType=StringType())
def weekdayStr(d):
    # calendar.day_name is a list of day names; d.weekday() returns 0-6
    return calendar.day_name[d.weekday()]

# UDF returning integer weekday number (0=Monday, 6=Sunday)
@udf(returnType=IntegerType())
def weekday(d):
    return d.weekday()

In [ ]:
# ✅ Exercise 8 Solution
# Apply UDF to extract weekday from dropoff datetime → group → count → sort
weekdayRows = (
    trips
    .select(weekdayStr(trips.tpep_dropoff_datetime).alias("weekday"))  # apply UDF
    .groupBy("weekday")                                                 # group by day name
    .count()                                                            # count trips per day
    .orderBy("count", ascending=False)                                  # sort by most trips first
    .collect()                                                          # bring to Python (small result)
)

barchart(weekdayRows, "weekday")
# Expected: Friday and Saturday have most trips; Monday and Tuesday fewer

# ✅ Exercise 9 — Trips per Hour of Day

**Task:** Count trips per hour (0–23) and visualize as a bar chart.

**Pattern to expect:**
- Morning rush: 7–9 AM peak
- Evening rush: 5–8 PM peak (bigger)
- Late night: 10 PM – 2 AM elevated (bars/restaurants)
- 3–5 AM: lowest (overnight)

In [ ]:
# ✅ Exercise 9 Solution
@udf(returnType=IntegerType())
def hour(d):
    return d.hour   # d is a datetime object; .hour gives 0-23

hourRows = (
    trips
    .select(hour(trips.tpep_dropoff_datetime).alias("hour"))  # extract hour from dropoff time
    .groupBy("hour")
    .count()
    .orderBy("count", ascending=False)
    .collect()
)

barchart(hourRows, "hour")
# Expected: evening hours (18-22) have most trips; 3-5 AM are fewest

# Map Visualisations

**leafmap** is a Python library for interactive maps.  
We will:
1. Show an interactive map of NYC
2. Colour each taxi zone by trip density (red = many trips)
3. Add a heat map overlay

In [ ]:
import leafmap

In [ ]:
def getMap():
    map_args = {
        "google_map": "HYBRID",
        "center": [40.702557, -74.012318],  # New York City centre
        "zoom": 12,
        "height": "450px",
        "width": "800px",
        "max_zoom": "20"
    }
    return leafmap.Map(**map_args)

In [ ]:
# Display the base map
getMap()

In [ ]:
def taxizoneColorFunction(taxiZonesIntensity, maximum_intensity, taxizoneFeature):
    """Map a taxi zone intensity (trip count) to a red color gradient."""
    taxizoneId = taxizoneFeature["properties"]["LocationID"]
    taxizoneIntensity = taxiZonesIntensity[taxizoneId] if taxizoneId in taxiZonesIntensity else 0
    return {
        "color": "black",
        # Red channel = 0-255 based on intensity; green/blue = 0
        "fillColor": '#%02X0000' % (int(taxizoneIntensity * 255 / maximum_intensity))
    }

def getTaxiZoneStylingFunction(taxiZonesIntensity):
    maximum_intensity = max(taxiZonesIntensity.values())
    return lambda x: taxizoneColorFunction(taxiZonesIntensity, maximum_intensity, x)

In [ ]:
taxizonesFile = base_directory + "/taxizonesdata/taxi_zones.shp"

def getZoneCenters():
    """Compute the centroid (avg lat, lon) for each taxi zone from the shapefile."""
    zone_centers = {}
    my_geojson = leafmap.shp_to_geojson(taxizonesFile)
    for feature in my_geojson["features"]:
        location = feature["properties"]["LocationID"]
        coordinates = feature["geometry"]["coordinates"]
        avg_lat = 0
        avg_lon = 0
        count = 0
        for coordinate_list in coordinates:
            for coordinate in coordinate_list:
                if type(coordinate) == tuple and len(coordinate) == 2:
                    avg_lat += coordinate[1]
                    avg_lon += coordinate[0]
                    count += 1
                elif len(coordinate) > 2:
                    for coord in coordinate:
                        avg_lat += coord[1]
                        avg_lon += coord[0]
                        count += 1
        avg_lat = avg_lat / count
        avg_lon = avg_lon / count
        zone_centers[location] = [avg_lat, avg_lon]
    return zone_centers

zoneCenters = getZoneCenters()

In [ ]:
def getHeatCenters(taxizoneIntensityMap):
    """Convert zone-intensity map to [lat, lon, intensity] list for heatmap."""
    heat_data = []
    for key, value in zoneCenters.items():
        location = key
        (lat, lon) = value
        taxizoneIntensity = taxizoneIntensityMap[location] if location in taxizoneIntensityMap else 0
        heat_data.append([lat, lon, taxizoneIntensity])
    return heat_data

# ✅ Exercise 10 — Trip counts per pickup and dropoff zone

**Task:** Count how many trips **start** and **end** in each taxi zone.  
Then display colored maps for pickups and dropoffs.

**Columns:** `PULocationID` = pickup zone, `DOLocationID` = dropoff zone  
Each is an integer ID corresponding to one of the 265 NYC taxi zones.

In [ ]:
# ✅ Exercise 10 Solution

# Count pickups per zone and collect into Python dict {zoneID -> count}
pickupData = (
    trips
    .groupBy("PULocationID")
    .count()
    .collect()
)

# Count dropoffs per zone
dropoffData = (
    trips
    .groupBy("DOLocationID")
    .count()
    .collect()
)

# Convert Spark Row objects to Python dicts
grouped_by_pickup_location  = {row["PULocationID"]: row["count"] for row in pickupData}
grouped_by_dropoff_location = {row["DOLocationID"]: row["count"] for row in dropoffData}

print(f"Pickup zones covered: {len(grouped_by_pickup_location)}")
print(f"Dropoff zones covered: {len(grouped_by_dropoff_location)}")

In [ ]:
# Pickup heatmap — darker red = more pickups
m = getMap()
m.add_shp(
    in_shp=taxizonesFile, layer_name="taxizone",
    style={}, hover_style={},
    style_callback=getTaxiZoneStylingFunction(grouped_by_pickup_location),
    fill_colors=None, info_mode='on_hover'
)
m.layer_opacity('taxizone', 0.9)
m.add_heatmap(data=getHeatCenters(grouped_by_pickup_location), name='pickup_heat', radius=10)
m.layer_opacity('pickup_heat', 0.9)
m

In [ ]:
# Dropoff heatmap — darker red = more dropoffs
m = getMap()
m.add_shp(
    in_shp=taxizonesFile, layer_name="taxizone",
    style={}, hover_style={},
    style_callback=getTaxiZoneStylingFunction(grouped_by_dropoff_location),
    fill_colors=None, info_mode='on_hover'
)
m.layer_opacity('taxizone', 0.9)
m.add_heatmap(data=getHeatCenters(grouped_by_dropoff_location), name='dropoff_heat', radius=10)
m.layer_opacity('dropoff_heat', 0.9)
m

# ✅ Exercise 11 — Top 10 highest-tip trips (excluding Unknown zones)

**Task:** Find the 10 trips with the highest tip amounts, **excluding trips whose pickup or dropoff zone is marked as 'Unknown'**.  
Then visualize those trips as lines on the map.

**Steps:**
1. Load `taxi_zone_lookup.csv` → contains `LocationID`, `Borough`, `Zone`
2. Find which LocationIDs have Borough == 'Unknown'
3. Join `trips` with `zoneLookup` twice (once for PU, once for DO)
4. Filter out Unknown zones
5. Sort by `tip_amount` descending, take top 10
6. Visualize

In [ ]:
# Load the zone lookup table (LocationID → Borough, Zone name)
zoneLookup = spark.read.csv(
    base_directory + "/taxizonesdata/taxi_zone_lookup.csv",
    header=True,
    inferSchema=True
)
zoneLookup.show(5)

In [ ]:
# Inspect unknown borough values
zoneLookup.filter(zoneLookup.Borough == "Unknown").show()
zoneLookup.filter(zoneLookup.Borough == "N/A").show()

In [ ]:
# Check data types to match join keys
print("zoneLookup dtypes:", zoneLookup.dtypes)
print("trips dtypes (location cols):", [c for c in trips.dtypes if 'location' in c[0].lower()])

In [ ]:
# ✅ Exercise 11 Solution

# Step 1: Get the set of LocationIDs that are "Unknown" borough
unknown_ids = (
    zoneLookup
    .filter(zoneLookup.Borough == "Unknown")
    .select("LocationID")
    .rdd.flatMap(lambda x: x)
    .collect()
)
print(f"Unknown LocationIDs: {unknown_ids}")

# Step 2: Create aliases for joining zoneLookup twice (PU and DO)
pu_lookup = zoneLookup.alias("pu").withColumnRenamed("LocationID", "PU_ID") \
                                   .withColumnRenamed("Borough", "PU_Borough")
do_lookup = zoneLookup.alias("do").withColumnRenamed("LocationID", "DO_ID") \
                                   .withColumnRenamed("Borough", "DO_Borough")

# Step 3: Join trips with zone lookups to get borough names
temporary = (
    trips
    .join(pu_lookup, trips.PULocationID == pu_lookup.PU_ID, how="left")
    .join(do_lookup, trips.DOLocationID == do_lookup.DO_ID, how="left")
    .filter(
        (col("PU_Borough") != "Unknown") &
        (col("DO_Borough") != "Unknown") &
        col("PU_Borough").isNotNull() &
        col("DO_Borough").isNotNull()
    )
)

# Step 4: Sort by tip amount descending, take top 10
# .orderBy(..., ascending=False) → sorted by tip desc
# .limit(10) → take top 10
# .collect() → pull to Python driver
tripsWithHighestTips = (
    temporary
    .orderBy(col("tip_amount"), ascending=False)
    .limit(10)
    .collect()
)

print(f"\nTop 10 trips by tip amount:")
for row in tripsWithHighestTips:
    print(f"  Tip: ${row['tip_amount']:.2f} | "
          f"From zone {row['PULocationID']} → zone {row['DOLocationID']} | "
          f"Distance: {row['trip_distance']:.1f} miles")

In [ ]:
# Convert trip list to GeoJSON LineString features (start → end zone centroids)
from geojson import FeatureCollection, Feature, LineString

def to_lon_and_lat(latLonCoordinate):
    return [latLonCoordinate[1], latLonCoordinate[0]]

def trip_to_geojson(trip):
    start_point = to_lon_and_lat(zoneCenters[trip["PULocationID"]])
    end_point   = to_lon_and_lat(zoneCenters[trip["DOLocationID"]])
    props = {
        "starttime": trip["tpep_pickup_datetime"].isoformat(),
        "startzone": trip["PULocationID"],
        "endtime":   trip["tpep_dropoff_datetime"].isoformat(),
        "endzone":   trip["DOLocationID"],
        "tip":       trip["tip_amount"],
    }
    return Feature(geometry=LineString([start_point, end_point]), properties=props)

def tripList_to_geojson(tripList):
    return FeatureCollection(list(map(trip_to_geojson, tripList)))

trip_geojson = tripList_to_geojson(tripsWithHighestTips)

In [ ]:
# Visualize the 10 highest-tip trips as red lines on the map
m = getMap()
m.add_shp(in_shp=taxizonesFile, layer_name="taxizone")
m.layer_opacity('taxizone', 0.9)
m.add_geojson(in_geojson=trip_geojson, layer_name="connections", style={"color": "red"})
m.layer_opacity('connections', 1.0)
m
# Each red line connects the pickup zone centroid to the dropoff zone centroid
# High-tip trips are often long-distance or to/from airports